In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · The same patterns with ADK 2 `Workflow` — worked

Google's ADK 2 gives you the primitives natively: **interrupts** (`RequestInput`), **resumability** (`ResumabilityConfig`), **replay-safe nodes** (`rerun_on_resume`), **routing** (`ctx.route`) and pluggable **session stores** (SQLite → Cloud SQL). This notebook runs the graph *without a model* so you can see the mechanics; swap `use_model=True` to put Gemini in the `plan` node.

```mermaid
flowchart LR
  S((START)) --> B[agree_budget<br/>RequestInput 'budget'<br/>rerun_on_resume=True]
  B --> P[plan<br/>judgement]
  P --> Q[queue_up<br/>side effect<br/>rerun_on_resume=False]
  Q --> C[check_front<br/>RequestInput 'wake'<br/>rerun_on_resume=True]
  C -- ready --> Y[buy<br/>idempotency key]
  C -- sold_out --> A[abandon]
```
(Cells use top-level `await` — Jupyter already runs an event loop.)

In [1]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [2]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from lragents.adk.nightly_workflow import ASK_BUDGET, WAKE, Venue, build_app, run_until_interrupt

venue = Venue()
svc = InMemorySessionService()                       # prod: DatabaseSessionService("postgresql+asyncpg://...") or VertexAiSessionService
runner = Runner(app=build_app(venue), session_service=svc)
session = await svc.create_session(app_name="nightly_app", user_id="anil",
                                   state={"events": [{"id": "ams-tue", "weekday": "Tuesday"}, {"id": "ams-sat", "weekday": "Saturday"}]})
async def state(s=svc, sid=None):
    return (await s.get_session(app_name="nightly_app", user_id="anil", session_id=sid or session.id)).state

r1 = await run_until_interrupt(runner, "anil", session.id, text="Get us two tickets")
print("stopped on:", r1["interrupts"], "| invocation:", r1["invocation_id"][:12], "| venue calls:", venue.calls)

stopped on: ['budget'] | invocation: e-bb35b2e0-1 | venue calls: []


The graph asked a question and **stopped**. The invocation lives in the session store; the Python process could exit here.

Answer it by resuming the *same invocation* with a `FunctionResponse` whose id is the interrupt id:

In [3]:
r2 = await run_until_interrupt(runner, "anil", session.id, invocation_id=r1["invocation_id"], answers={ASK_BUDGET: {"budget": 250}})
print("stopped on:", r2["interrupts"], "| venue calls:", venue.calls, "| state:", {k: v for k, v in (await state()).items() if k != "events"})

stopped on: ['wake'] | venue calls: ['join_queue'] | state: {'budget_per_seat': 250.0, 'event_id': 'ams-sat', 'ticket': 'q_fd7cfea5-4629-4486-aa77-7e080cefa139:ams-sat'}


Four nodes ran off one answer: `plan` chose Saturday (weekend rule), `queue_up` took **one** ticket, `check_front` found us at #14,203 and parked on `wake`.

## Wake-ups (what Cloud Scheduler → Pub/Sub → `/wake` does)

In [4]:
ticket = (await state())["ticket"]
venue.advance(ticket, 10_000)                        # the queue moves while nothing of ours runs
r3 = await run_until_interrupt(runner, "anil", session.id, invocation_id=r2["invocation_id"], answers={WAKE: {"ok": True}})
print("wake 1 → stopped on:", r3["interrupts"], "| join_queue calls:", venue.calls.count("join_queue"), "| position:", venue.position(ticket))
venue.advance(ticket, 10_000)
r4 = await run_until_interrupt(runner, "anil", session.id, invocation_id=r3["invocation_id"], answers={WAKE: {"ok": True}})
print("wake 2 → stopped on:", r4["interrupts"], "| venue calls:", venue.calls)
print("order:", (await state())["order"])

wake 1 → stopped on: ['wake'] | join_queue calls: 1 | position: 4203
wake 2 → stopped on: [] | venue calls: ['join_queue', 'purchase']
order: {'order_id': 'ord_1', 'event_id': 'ams-sat', 'seats': 2, 'replayed': False}


`queue_up` never re-ran across four wake-ups (`rerun_on_resume=False`); `check_front` re-ran every time (`True`); `buy` executed once with an idempotency key.

## Staleness guard: the world changed while we waited

In [5]:
venue2 = Venue(); svc2 = InMemorySessionService(); runner2 = Runner(app=build_app(venue2), session_service=svc2)
s2 = await svc2.create_session(app_name="nightly_app", user_id="anil", state={"events": [{"id": "ams-sat", "weekday": "Saturday"}]})
a = await run_until_interrupt(runner2, "anil", s2.id, text="go")
b = await run_until_interrupt(runner2, "anil", s2.id, invocation_id=a["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
t2 = (await state(svc2, s2.id))["ticket"]
venue2.advance(t2, 99_999); venue2.sell_out("ams-sat")
await run_until_interrupt(runner2, "anil", s2.id, invocation_id=b["invocation_id"], answers={WAKE: {"ok": True}})
print("calls:", venue2.calls, "| orders:", venue2.orders, "| routed to abandon:", (await state(svc2, s2.id))["order"] is None)

calls: ['join_queue'] | orders: {} | routed to abandon: True


## The classic mistake: a new invocation instead of a resume
Typing in the chat box (no `invocation_id`) starts a **new** invocation → the graph replays from START → a second queue ticket.

In [6]:
venue3 = Venue(); svc3 = InMemorySessionService(); runner3 = Runner(app=build_app(venue3), session_service=svc3)
s3 = await svc3.create_session(app_name="nightly_app", user_id="anil", state={"events": [{"id": "ams-sat", "weekday": "Saturday"}]})
a = await run_until_interrupt(runner3, "anil", s3.id, text="go")
await run_until_interrupt(runner3, "anil", s3.id, invocation_id=a["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
c = await run_until_interrupt(runner3, "anil", s3.id, text="where am I in line?")       # WRONG: new invocation
await run_until_interrupt(runner3, "anil", s3.id, invocation_id=c["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
print("join_queue calls:", venue3.calls.count("join_queue"), "← two tickets")

join_queue calls: 2 ← two tickets


## What the session store holds
Every event (interrupts included) is a row; this is what survives a restart when the store is Cloud SQL.

In [7]:
sess = await svc.get_session(app_name="nightly_app", user_id="anil", session_id=session.id)
print("events:", len(sess.events), "| invocations:", len({e.invocation_id for e in sess.events}))
for e in sess.events[:12]:
    fc = [p.function_call.name for p in (e.content.parts if e.content else []) or [] if p.function_call]
    print(f"  {e.invocation_id[:10]}  author={e.author:<8} fc={fc} state_delta={list((e.actions.state_delta or {}).keys()) if e.actions else []}")

events: 36 | invocations: 1
  e-bb35b2e0  author=user     fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=['adk_request_input'] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=user     fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=['budget_per_seat']
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=['event_id']
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]
  e-bb35b2e0  author=nightly  fc=[] state_delta=[]


## Deploying this shape on GCP
| Local | Cloud | Change |
|---|---|---|
| `InMemorySessionService` | Cloud SQL (Postgres) via `DatabaseSessionService("postgresql+asyncpg://...?host=/cloudsql/...")` or Agent Runtime Sessions | a connection string |
| `adk web` | one Cloud Run service (`src/lragents/adk/main.py`) | a Dockerfile + entrypoint |
| you calling `run_until_interrupt` | Cloud Scheduler → Pub/Sub → push → `POST /wake` | `trigger_sources=["pubsub"]` + custom `/wake` |
| `Venue` | the real API with an `Idempotency-Key` header | — |

Note the codelab's warning that ADK's built-in Pub/Sub trigger route **creates a new session per message** — for long-running runs you need a wake endpoint that resumes an *existing* session, which is what `main.py` adds.

## Takeaways
* `RequestInput` is one primitive for both "waiting for a person" and "waiting for the world".
* `rerun_on_resume` is the ADK spelling of *idempotent vs re-check* — decide it per node, deliberately.
* Resume the invocation; don't start a new one.
* Prompts can't wake themselves; something with a clock has to call the endpoint.